In [3]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from tqdm import tqdm

df = pd.read_csv("simple_synthetic_sales_data.csv")
df["date"] = pd.to_datetime(df["date"])

books_df = df[df["category"] == "Books"].copy()

# -----------------------------
# Feature Engineering
# -----------------------------
def create_features(df):
    df = df.copy()
    
    df["dow"] = df["date"].dt.weekday
    df["month"] = df["date"].dt.month
    
    df["lag_7"] = df["sales"].shift(7)
    df["lag_14"] = df["sales"].shift(14)
    df["lag_30"] = df["sales"].shift(30)
    
    df["rolling_7"] = df["sales"].rolling(7).mean()
    
    df["target_7d"] = df["sales"].shift(-1).rolling(7).sum()
    
    return df

# -----------------------------
# Bootstrap Forecast
# -----------------------------
def bootstrap_forecast(df_feat, features, n_models=50):
    
    X = df_feat[features]
    y = df_feat["target_7d"]
    
    preds = []
    
    for _ in range(n_models):
        # bootstrap sample
        idx = np.random.choice(len(X), size=len(X), replace=True)
        X_sample = X.iloc[idx]
        y_sample = y.iloc[idx]
        
        model = XGBRegressor(
            n_estimators=300,
            max_depth=5,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8
        )
        
        model.fit(X_sample, y_sample)
        
        pred = model.predict(X.iloc[[-1]])[0]
        preds.append(pred)
    
    preds = np.array(preds)
    
    return {
        "forecast_mean": preds.mean(),
        "lower_80": np.percentile(preds, 10),
        "upper_80": np.percentile(preds, 90),
        "lower_95": np.percentile(preds, 2.5),
        "upper_95": np.percentile(preds, 97.5),
    }

# -----------------------------
# Run
# -----------------------------
results = []

for product, group in tqdm(books_df.groupby("product")):
    
    df_feat = create_features(group).dropna()
    
    features = [
        "dow", "month", "promo", "price",
        "lag_7", "lag_14", "lag_30", "rolling_7"
    ]
    
    res = bootstrap_forecast(df_feat, features)
    
    results.append({
        "product": product,
        **res
    })

results_df = pd.DataFrame(results)

top3 = results_df.sort_values("forecast_mean", ascending=False).head(3)

print("\nTop-3 Products in Books (XGBoost + Bootstrap Uncertainty):\n")
print(top3.round(2))

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 6/6 [02:26<00:00, 24.40s/it]


Top-3 Products in Books (XGBoost + Bootstrap Uncertainty):

     product  forecast_mean  lower_80  upper_80  lower_95  upper_95
3      Novel    1573.199951   1563.18   1591.78   1555.98   1606.69
1     Comics    1512.900024   1437.14   1555.60   1424.37   1556.68
0  Biography    1325.119995   1309.67   1338.14   1298.10   1351.79
